# Reading And Writing A ROOT File

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

Every analysis starts the same way: a ROOT file written by GATE, and the
question of what is inside it. This notebook opens such a file, finds out which
trees and branches it holds, reads the part of it an analysis needs, keeps the
rows that matter, and writes them to a new ROOT file that the next step of the
analysis can read.

It also shows the two places where the round trip surprises: a file holding a
selection of branches is no longer a whole "Hits" tree, and an array branch
such as `volumeID` does not survive a detour through pandas.

## What will you learn from this notebook

- how to open a GATE ROOT file and list the trees and branches it holds
- how to name a tree by its enum member or by its name as text
- how to read chosen branches of a tree, and what comes back
- how to tell a missing tree or branch from any other failure
- how to select rows in pandas and write them to a ROOT file
- how to read your own output back, and why the structure check has to be
  turned off for it
- how to keep an array branch intact while selecting rows

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

Two paths: the one place this notebook may write, and the example file it
reads. The file is a small GATE output carried in the repository; what it holds
is described in [`../data/README.md`](../data/README.md).

In [2]:
output = paths.output_dir("tutorial-02-reading-writing-root-file")
example_input = paths.input_example("positronium-source-all-variants.root")

Everything this notebook uses is imported from the package root. The
submodules are an implementation detail and may move between versions.

In [3]:
from opengate_gate_tree import (
    GammaType,
    GateTree,
    GateTreeError,
    OutputFileFormat,
    RootFile,
    TreeData,
    TreeNotFoundError,
    parse_gate_tree,
    read_tree,
    write_tree,
)

### Opening A File

`RootFile` opens the file once and keeps it open, so that several questions —
and several trees — cost one opening. In a script it belongs in a `with` block,
which closes the file however the block ends. A notebook asks its questions one
cell at a time, so here the file is opened in one cell and closed explicitly at
the end of the section.

In [4]:
root_file = RootFile(example_input)

### What The File Holds

The trees in the file, with the ROOT cycle suffix (`Hits;1`) removed. Only
trees are listed: histograms GATE stores beside them are not something this
package reads.

In [5]:
root_file.tree_names

('Hits',)

A tree is named by a member of `GateTree`. `parse_gate_tree` turns a name
given as text — from a configuration file or a command line — into that member,
so both forms ask the same question:

In [6]:
root_file.has_tree(GateTree.HITS), root_file.has_tree(parse_gate_tree("Hits"))

(True, True)

In [7]:
root_file.has_tree(GateTree.SINGLES)

False

The simulation wrote hits only: no digitizer ran, so there is no `Singles`
tree. The branches of the `Hits` tree are the ones the file actually holds, in
the order it holds them — not a list built into the package:

In [8]:
root_file.branch_names(GateTree.HITS)

('PDGEncoding',
 'trackID',
 'parentID',
 'trackLocalTime',
 'time',
 'edep',
 'stepLength',
 'trackLength',
 'posX',
 'posY',
 'posZ',
 'localPosX',
 'localPosY',
 'localPosZ',
 'sourcePosX',
 'sourcePosY',
 'sourcePosZ',
 'sourceID',
 'eventID',
 'runID',
 'volumeID',
 'processName',
 'momDirX',
 'momDirY',
 'momDirZ',
 'photonID',
 'nPhantomCompton',
 'nCrystalCompton',
 'nPhantomRayleigh',
 'nCrystalRayleigh',
 'nInteractions',
 'primaryID',
 'axialPos',
 'rotationAngle',
 'comptVolName',
 'RayleighVolName',
 'sourceType',
 'decayType',
 'gammaType',
 'decayIndex')

### Reading A Tree

An analysis rarely needs forty branches. Naming the ones it does keeps the
memory down, and fixes the order of the columns:

In [9]:
hits = root_file.read(
    GateTree.HITS,
    branches=["eventID", "trackID", "edep", "gammaType", "processName"],
)
hits

TreeData(tree='Hits', entries=500, branches=5)

What comes back is a `TreeData`: the branches as NumPy arrays, under the
names they have in the file. It says how many entries it holds, which branches,
and of what type — the types GATE wrote, unchanged:

In [10]:
hits.entry_count, hits.branch_names

(500, ('eventID', 'trackID', 'edep', 'gammaType', 'processName'))

In [11]:
dict(hits.dtypes)

{'eventID': dtype('int32'),
 'trackID': dtype('int32'),
 'edep': dtype('float32'),
 'gammaType': dtype('int32'),
 'processName': dtype('O')}

In [12]:
hits["edep"][:5]

array([0.09453767, 0.41646123, 0.2204028 , 0.2905961 , 0.1327286 ],
      dtype=float32)

Asking for something the file does not hold fails with an error that names
what it does hold, and the file it looked in. Every failure of reading or writing derives from
`GateTreeError`, so one clause covers them all, and a more specific one comes
first when a case deserves its own answer:

In [13]:
try:
    root_file.read(GateTree.SINGLES)
except TreeNotFoundError:
    print(f"no Singles tree; the file holds {root_file.tree_names}")
except GateTreeError as error:
    print(f"could not read the file: {error}")

no Singles tree; the file holds ('Hits',)


### From TreeData To pandas

Selecting rows, grouping them and plotting them is what pandas is for.
`to_dataframe()` hands the columns over, one per branch:

In [14]:
frame = hits.to_dataframe()
frame

,eventID,trackID,edep,gammaType,processName
0,0,2,0.094538,2,Compton
1,0,2,0.416461,2,PhotoElectric
2,0,1,0.220403,2,Compton
3,0,1,0.290596,2,PhotoElectric
4,1,2,0.132729,2,Compton
...,...,...,...,...,...
495,113,1,0.033188,0,Compton
496,113,1,0.225968,0,PhotoElectric
497,114,3,0.505492,2,PhotoElectric
498,114,2,0.099978,2,PhotoElectric


### Choosing The Rows

The selection to keep: hits of annihilation gammas that deposited more than
200 keV. `edep` is in MeV, as GATE writes it, and `GammaType.ANNIHILATION` is
the integer GATE wrote, so both comparisons run on the columns as they were
read.

In [15]:
selected = frame[(frame["gammaType"] == GammaType.ANNIHILATION) & (frame["edep"] > 0.2)]
selected

,eventID,trackID,edep,gammaType,processName
1,0,2,0.416461,2,PhotoElectric
2,0,1,0.220403,2,Compton
3,0,1,0.290596,2,PhotoElectric
7,1,2,0.273780,2,PhotoElectric
8,1,1,0.510999,2,PhotoElectric
...,...,...,...,...,...
482,111,1,0.313066,2,PhotoElectric
484,112,4,0.243170,2,PhotoElectric
486,112,3,0.306889,2,PhotoElectric
487,112,2,0.260496,2,PhotoElectric


### Writing A ROOT File

`TreeData.from_dataframe` turns the frame back into the package representation:
every column becomes a branch, and the index — which still carries the row
numbers of the original tree — is dropped. The tree it is given decides the
name of the `TTree` in the output file.

In [16]:
selected_hits = TreeData.from_dataframe(GateTree.HITS, selected)
selected_hits

TreeData(tree='Hits', entries=227, branches=5)

`write_tree` writes it in the format it is asked for and returns the path it
wrote to. The parent directory is made when missing, and an existing file is
overwritten without a prompt.

In [17]:
written = write_tree(selected_hits, output / "selected-hits.root", OutputFileFormat.ROOT)
written.relative_to(paths.TUTORIALS_DIR)

PosixPath('notebooks/.tmp/tutorial-02-reading-writing-root-file/selected-hits.root')

The output is an ordinary ROOT file with one `TTree` named `Hits`, so C++
code using ROOT reads it with `TFile` and `TTree::SetBranchAddress`, and
`RDataFrame` works on it as well. The histograms GATE stored beside the trees
of the input file are not copied over.

In [18]:
with RootFile(written) as written_file:
    print(written_file.tree_names)
    print(written_file.branch_names(GateTree.HITS))

('Hits',)
('eventID', 'trackID', 'edep', 'gammaType', 'processName')


### Reading Your Own Output Back

Reading a "Hits" tree first recognises which structure it has and checks the
tree against it. The file just written holds five branches out of forty, which
is no longer any structure GATE writes — so the check refuses it, and says
why:

In [19]:
try:
    read_tree(written, GateTree.HITS)
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

UnknownHitsVariantError: Tree 'Hits' does not hold a branch that identifies a known Hits structure: neither 'volumeID' nor 'volumeID[0]' is present. Branches present: ['eventID', 'trackID', 'edep', 'gammaType', 'processName'].


The file is not broken; it is a selection, and the check is meant for files
GATE wrote. `validate=False` reads it as it is:

In [20]:
read_back = read_tree(written, GateTree.HITS, validate=False)
read_back

TreeData(tree='Hits', entries=227, branches=5)

Nothing was lost on the way: the rows, the values and the types that come
back are the ones that were selected. The index of the selection is reset
before comparing, because the written tree numbers its entries from zero.

In [21]:
read_back.to_dataframe().equals(selected.reset_index(drop=True))

True

### Keeping An Array Branch

`volumeID` is not one number per hit but ten: GATE writes it as a fixed-width
array, and the package reads it as a two-dimensional column.

In [22]:
volumes = root_file.read(GateTree.HITS, ["eventID", "edep", "volumeID"])
dict(volumes.array_branches), volumes["volumeID"].shape

({'volumeID': 10}, (500, 10))

A pandas cell holds one value, so `to_dataframe()` spreads the array over ten
columns, `volumeID_0` to `volumeID_9`:

In [23]:
volumes.to_dataframe().columns.tolist()

['eventID',
 'edep',
 'volumeID_0',
 'volumeID_1',
 'volumeID_2',
 'volumeID_3',
 'volumeID_4',
 'volumeID_5',
 'volumeID_6',
 'volumeID_7',
 'volumeID_8',
 'volumeID_9']

`from_dataframe` does not fold them back: it cannot tell ten columns that
were once an array from ten that never were. A tree that went through pandas
is written with ten scalar branches instead of one array branch:

In [24]:
dict(TreeData.from_dataframe(GateTree.HITS, volumes.to_dataframe()).array_branches)

{}

When the array has to survive, select the rows on the arrays themselves. A
mask built on one branch is applied to every branch of the `TreeData`, and a
new `TreeData` is made from the result — the two-dimensional column keeps its
shape:

In [25]:
energetic = volumes["edep"] > 0.2

energetic_volumes = TreeData(
    GateTree.HITS,
    {name: volumes[name][energetic] for name in volumes.branch_names},
)
energetic_volumes, dict(energetic_volumes.array_branches)

(TreeData(tree='Hits', entries=326, branches=3), {'volumeID': 10})

In [26]:
written_volumes = write_tree(
    energetic_volumes, output / "energetic-volumes.root", OutputFileFormat.ROOT
)
dict(read_tree(written_volumes, GateTree.HITS, validate=False).array_branches)

{'volumeID': 10}

The ROOT file holds `volumeID` as an array again, and C++ code reads it into
an `Int_t[10]`.

### Closing The File

Everything this notebook needed has been read, so the file is closed. In a
script the `with` block does this on its own.

In [27]:
root_file.close()

## Summary

A GATE ROOT file is inspected and read through `RootFile`, which says what
trees and branches it holds and hands the chosen ones over as a `TreeData`;
pandas selects the rows, and `write_tree` writes them to a new ROOT file. A
file written that way is read back with `validate=False`, and an array branch
such as `volumeID` keeps its shape only when the rows are selected on the
`TreeData` rather than on the frame.

## Bibliography

- [Loading files](https://opengate-gate-tree.readthedocs.io/en/latest/guide/loading.html)
  — how a tree is found, which branch types are read, and what reading checks
- [Exporting files](https://opengate-gate-tree.readthedocs.io/en/latest/guide/export.html)
  — what each output format preserves, and reading your own output back
- [Using the package as a library](https://opengate-gate-tree.readthedocs.io/en/latest/guide/library.html)
  — the public API and the errors it raises
- [Data representations](https://opengate-gate-tree.readthedocs.io/en/latest/guide/representations.html)
  — `TreeData`, its pandas view, and how array branches are expanded
- [The "Hits" Tree](https://opengate-gate-tree.readthedocs.io/en/latest/guide/hits.html)
  — the structures the check recognises
- [Units](https://opengate-gate-tree.readthedocs.io/en/latest/guide/units.html)
  — why `edep` is in MeV

## Changelog

- v1: written.